In [15]:
import pandas as pd

df = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=0)
df["StockCode"] = df["StockCode"].astype(str)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  str           
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(2), str(2)
memory usage: 32.1+ MB


In [17]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845018,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


In [18]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [19]:
print("Przed czyszczeniem:", len(df))

# 1. Usuń wiersze bez Customer ID - nie da się rekomendować anonimowym klientom
df_clean = df.dropna(subset=["Customer ID"])
print("Po usunięciu braków Customer ID:", len(df_clean))

# 2. Usuń zwroty (ujemna ilość = zwrot towaru, nie zakup)
df_clean = df_clean[df_clean["Quantity"] > 0]
print("Po usunięciu zwrotów:", len(df_clean))

# 3. Usuń błędne/zerowe ceny
df_clean = df_clean[df_clean["Price"] > 0]
print("Po usunięciu błędnych cen:", len(df_clean))

# 4. Konwersja Customer ID na int (był float przez obecność NaN, teraz ich nie ma)
df_clean["Customer ID"] = df_clean["Customer ID"].astype(int)

# 5. Reset indeksu (po usuwaniu wierszy zostają "dziury" w numeracji)
df_clean = df_clean.reset_index(drop=True)

# 6. Usuń całkowicie zduplikowane wiersze
df_clean = df_clean.drop_duplicates()
print("Po usunięciu duplikatów:", len(df_clean))

df_clean.info()

Przed czyszczeniem: 525461
Po usunięciu braków Customer ID: 417534
Po usunięciu zwrotów: 407695
Po usunięciu błędnych cen: 407664
Po usunięciu duplikatów: 400916
<class 'pandas.DataFrame'>
Index: 400916 entries, 0 to 407663
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      400916 non-null  object        
 1   StockCode    400916 non-null  str           
 2   Description  400916 non-null  object        
 3   Quantity     400916 non-null  int64         
 4   InvoiceDate  400916 non-null  datetime64[us]
 5   Price        400916 non-null  float64       
 6   Customer ID  400916 non-null  int64         
 7   Country      400916 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(2), object(2), str(2)
memory usage: 27.5+ MB


In [20]:
customer_product_matrix = df_clean.pivot_table(
    index="Customer ID",
    columns="StockCode",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

customer_product_matrix.shape

(4312, 4017)

In [21]:
total_cells = customer_product_matrix.size
zero_cells = (customer_product_matrix == 0).sum().sum()
sparsity = zero_cells / total_cells * 100

print(f"Procent zer w macierzy: {sparsity:.2f}%")

Procent zer w macierzy: 98.42%


In [22]:
from sklearn.metrics.pairwise import cosine_similarity

product_similarity = cosine_similarity(customer_product_matrix.T)

In [23]:
product_similarity_df = pd.DataFrame(
    product_similarity,
    index=customer_product_matrix.columns,
    columns=customer_product_matrix.columns
)

def get_similar_products(product_id: str, n: int = 5) -> pd.Series:
    posortowane = product_similarity_df[product_id].sort_values(ascending=False)
    return posortowane.iloc[1:n + 1]

In [24]:
product_names = (
    df_clean[["StockCode", "Description"]]
    .drop_duplicates(subset="StockCode")
    .set_index("StockCode")
)

recommendations = get_similar_products("10002", n=5)
recommendations_with_names = recommendations.to_frame("similarity_score").join(product_names)

In [25]:
import numpy as np

np.random.seed(42)

test_rows = []

for customer_id in df_clean["Customer ID"].unique():
    customer_data = df_clean[df_clean["Customer ID"] == customer_id]

    if len(customer_data) >= 2:
        test_row = customer_data.sample(n=1, random_state=42)
        test_rows.append(test_row)

test_set = pd.concat(test_rows)
train_set = df_clean.drop(test_set.index)

print("Train set:", len(train_set))
print("Test set:", len(test_set))

Train set: 396695
Test set: 4221


In [26]:
train_matrix = train_set.pivot_table(
    index="Customer ID",
    columns="StockCode",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

train_similarity = cosine_similarity(train_matrix.T)

train_similarity_df = pd.DataFrame(
    train_similarity,
    index=train_matrix.columns,
    columns=train_matrix.columns
)

train_matrix.shape

(4312, 4015)

In [28]:
produkty_oryginalne = set(customer_product_matrix.columns)
produkty_treningowe = set(train_matrix.columns)

zniknięte_produkty = produkty_oryginalne - produkty_treningowe
print(zniknięte_produkty)
print(len(zniknięte_produkty))

{'44242A', '20885'}
2


In [29]:
print("44242A:", (df_clean["StockCode"] == "44242A").sum())
print("20885:", (df_clean["StockCode"] == "20885").sum())

44242A: 1
20885: 1
